
# Face Dataset Inspection

## Objective
The objective of this notebook is to inspect and analyze
the raw facial image dataset before applying face detection
and preprocessing techniques.

## Dataset Description
- User_01: 60 images
- User_02: 62 images
- Total: 122 images

## Inspection Tasks
1. Dataset structure validation.
2. Image integrity checking.
3. Image dimensions analysis.
4. Color channel analysis.
5. Image quality assessment.
6. Duplicate image detection.
7. Visual inspection.

## Dataset Location
Dataset/Raw/Face/


In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


## 1. Dataset structure validation.

In [ ]:

PROJECT_ROOT = Path.cwd().parent.parent

DATASET_PATH = PROJECT_ROOT / "Dataset" / "Raw" / "Face"

USERS = ["User_01", "User_02"]

print("Project Root:", PROJECT_ROOT)
print("Dataset Path:", DATASET_PATH)
print("Dataset Exists:", DATASET_PATH.exists())


## 2. Image integrity checking.

In [ ]:
for user in USERS:
    user_path = DATASET_PATH / user

    if not user_path.exists():
        print(f"{user}: Folder Not Found")
        continue

    images = [
        file for file in user_path.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    print(f"{user}: {len(images)} images")

## 3. Image dimensions analysis.

In [ ]:
# ==========================================
# Dataset Inspection Report
# ==========================================

# 1. General Statistics
print("=" * 60)
print("DATASET INSPECTION REPORT")
print("=" * 60)

print(f"\nTotal Images: {len(image_df)}")

print("\nImages Per User:")
print(image_df["User"].value_counts().to_string())

# 2. Image Integrity
print("\n" + "=" * 60)
print("IMAGE INTEGRITY")
print("=" * 60)

print(image_df["Status"].value_counts().to_string())

# 3. Image Dimensions
print("\n" + "=" * 60)
print("IMAGE DIMENSIONS DISTRIBUTION")
print("=" * 60)

print(image_df["Dimensions"].value_counts().to_string())


# 4. Visual Inspection
fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for row, user in enumerate(USERS):

    user_path = DATASET_PATH / user

    image_files = sorted([
        file for file in user_path.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    for col in range(4):

        image_path = image_files[col]

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        axes[row, col].imshow(image)
        axes[row, col].set_title(
            f"{user}\n{image_path.name}",
            fontsize=9
        )
        axes[row, col].axis("off")

plt.suptitle("Visual Inspection - Sample Images", fontsize=15)
plt.tight_layout()
plt.show()

## 4. Color channel analysis.

In [ ]:
# ==========================================
# Brightness & Blur Analysis
# ==========================================

quality_results = []

for user in USERS:

    user_path = DATASET_PATH / user

    image_files = sorted([
        file for file in user_path.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    for image_path in image_files:

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        # Convert image to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Calculate brightness
        brightness = np.mean(gray)

        # Calculate blur score
        blur_score = cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()

        quality_results.append({
            "User": user,
            "Filename": image_path.name,
            "Brightness": round(brightness, 2),
            "Blur Score": round(blur_score, 2)
        })


# Create DataFrame
quality_df = pd.DataFrame(quality_results)


# ==========================================
# Statistical Summary
# ==========================================

print("=" * 60)
print("IMAGE QUALITY ANALYSIS")
print("=" * 60)

print("\nBrightness Statistics:")
print(quality_df["Brightness"].describe().round(2))

print("\nBlur Score Statistics:")
print(quality_df["Blur Score"].describe().round(2))

print("\nQuality Statistics Per User:")
display(
    quality_df.groupby("User")[
        ["Brightness", "Blur Score"]
    ].agg(["mean", "min", "max"]).round(2)
)


# ==========================================
# Visualization
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Brightness Distribution
axes[0].hist(
    quality_df["Brightness"],
    bins=15,
    edgecolor="black"
)
axes[0].set_title("Brightness Distribution")
axes[0].set_xlabel("Brightness")
axes[0].set_ylabel("Number of Images")

# Blur Distribution
axes[1].hist(
    quality_df["Blur Score"],
    bins=15,
    edgecolor="black"
)
axes[1].set_title("Blur Score Distribution")
axes[1].set_xlabel("Variance of Laplacian")
axes[1].set_ylabel("Number of Images")

plt.tight_layout()
plt.show()

## 5. Image quality assessment.

In [ ]:
# ==========================================
# Blur Score Investigation
# ==========================================

print("=" * 60)
print("LOWEST BLUR SCORES")
print("=" * 60)

display(
    quality_df.nsmallest(5, "Blur Score")[
        ["User", "Filename", "Brightness", "Blur Score"]
    ]
)

print("\n" + "=" * 60)
print("HIGHEST BLUR SCORES")
print("=" * 60)

display(
    quality_df.nlargest(5, "Blur Score")[
        ["User", "Filename", "Brightness", "Blur Score"]
    ]
)

In [ ]:
# ==========================================
# Visual Investigation of Blur Scores
# ==========================================

lowest_blur = quality_df.nsmallest(5, "Blur Score")
highest_blur = quality_df.nlargest(5, "Blur Score")

selected_images = pd.concat(
    [lowest_blur, highest_blur]
).reset_index(drop=True)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for i, row in selected_images.iterrows():

    image_path = (
        DATASET_PATH / row["User"] / row["Filename"]
    )

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax = axes[i // 5, i % 5]

    ax.imshow(image)
    ax.set_title(
        f'{row["User"]}\n'
        f'{row["Filename"]}\n'
        f'Blur: {row["Blur Score"]}',
        fontsize=9
    )
    ax.axis("off")

axes[0, 0].set_ylabel("Lowest Scores", fontsize=12)
axes[1, 0].set_ylabel("Highest Scores", fontsize=12)

plt.suptitle(
    "Visual Comparison of Lowest and Highest Blur Scores",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Blur Filtering & Dataset Organization
# ==========================================

import shutil

# Define paths
PROCESSED_PATH = PROJECT_ROOT / "Dataset" / "Processed" / "Face"

FILTERED_BLUR_PATH = PROCESSED_PATH / "Filtered_Blur"

# Images identified as blurry
excluded_images = {
    "User_01_055.jpg",
    "User_01_056.jpg",
    "User_01_012.jpg",
    "User_01_052.jpg",
    "User_01_013.jpg"
}

# Process each user
for user in USERS:

    source_folder = DATASET_PATH / user

    # Create the main processed folder
    processed_user_path = PROCESSED_PATH / user
    processed_user_path.mkdir(parents=True, exist_ok=True)

    copied_count = 0
    filtered_count = 0

    # Process images
    for image_path in source_folder.iterdir():

        if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        if image_path.name in excluded_images:

            # Create filtered folder only when needed
            filtered_user_path = FILTERED_BLUR_PATH / user
            filtered_user_path.mkdir(parents=True, exist_ok=True)

            shutil.copy2(
                image_path,
                filtered_user_path / image_path.name
            )

            filtered_count += 1

        else:

            # Copy accepted images to Processed
            shutil.copy2(
                image_path,
                processed_user_path / image_path.name
            )

            copied_count += 1

    print(f"\n{user}")
    print("Processed Images:", copied_count)
    print("Filtered Blur Images:", filtered_count)

print("\nDataset Organization Completed!")

## 6. Duplicate image detection.

In [ ]:
# ==========================================
# Exact Duplicate Images Detection
# ==========================================

import hashlib

PROCESSED_PATH = PROJECT_ROOT / "Dataset" / "Processed" / "Face"

image_hashes = []

for user in USERS:

    user_path = PROCESSED_PATH / user

    for image_path in user_path.iterdir():

        if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        with open(image_path, "rb") as image_file:
            image_hash = hashlib.md5(image_file.read()).hexdigest()

        image_hashes.append({
            "User": user,
            "Filename": image_path.name,
            "MD5": image_hash
        })


# Create DataFrame
hash_df = pd.DataFrame(image_hashes)

# Find exact duplicates
duplicate_df = hash_df[
    hash_df.duplicated(subset="MD5", keep=False)
].sort_values("MD5")


# Display Results
print("=" * 60)
print("EXACT DUPLICATE DETECTION")
print("=" * 60)

print("\nTotal Processed Images:", len(hash_df))
print("Unique Images:", hash_df["MD5"].nunique())
print("Duplicate Image Files:", len(duplicate_df))

if duplicate_df.empty:
    print("\nNo Exact Duplicate Images Found!")
else:
    print("\nDuplicate Images:")
    display(duplicate_df)